# CV HW: DETR на COCO-subset + синтетика через ControlNet

Ноутбук сделан как сценарий для запуска и защиты работы. Основной код лежит в `src/`, а здесь собраны команды, проверки, визуализации и короткие выводы.

Что должно получиться в конце:

- обученный DETR на COCO-subset из 10 классов;
- TensorBoard-логи, чекпойнт и profiler trace;
- графики `loss_ce`, `loss_bbox`, `loss_giou`;
- таблица `mAP / mAP50`;
- галерея предсказаний bbox;
- error analysis: classification / localization / missed / false positive;
- ablation для синтетики: `real` vs `real + synthetic`.

> Для быстрой демонстрации можно сначала прогнать toy-режим. Для настоящих метрик нужен COCO 2017 и GPU.

## 0. Проверка GPU

Для DETR желательно включить Colab GPU: `Runtime → Change runtime type → T4 / L4 / A100`.

На CPU можно проверить структуру проекта, но нормальное обучение будет очень медленным.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

import torch

print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM, GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print('GPU не видна. Для toy-проверки нормально, для обучения лучше включить GPU.')

## 1. Загрузка проекта

Есть два удобных варианта:

1. если проект уже на GitHub — вставить ссылку в `REPO_URL`;
2. если проекта ещё нет на GitHub — загрузить архив `.zip` вручную в Colab.

Для защиты обычно удобнее GitHub: преподаватель видит репозиторий, а ноутбук просто воспроизводит запуск.

In [ ]:
# ========= настройки =========
USE_GITHUB = False
REPO_URL = 'https://github.com/<your-login>/<your-repo>.git'  # поменять после загрузки на GitHub
PROJECT_DIR = Path('/content/cv_hw_detr_controlnet_repo')
ZIP_NAME_HINT = 'cv_hw_detr_controlnet_repo_student_App.zip'
# =============================

if PROJECT_DIR.exists():
    print('Проект уже есть:', PROJECT_DIR)
elif USE_GITHUB:
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    from google.colab import files
    print('Загрузи архив проекта, например:', ZIP_NAME_HINT)
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.endswith('.zip')]
    if not zip_names:
        raise RuntimeError('Не найден .zip архив проекта')
    zip_path = zip_names[0]
    !unzip -q -o "{zip_path}" -d /content
    # архив уже содержит папку cv_hw_detr_controlnet_repo
    if not PROJECT_DIR.exists():
        # на случай, если папка называется чуть иначе
        candidates = list(Path('/content').glob('*detr*repo*'))
        if candidates:
            PROJECT_DIR = candidates[0]
    print('Проект распакован:', PROJECT_DIR)

%cd {PROJECT_DIR}
!find . -maxdepth 2 -type f | sort | sed 's#^./##' | head -80

## 2. Установка зависимостей

В Colab уже есть `torch`, поэтому установка обычно занимает несколько минут. Если Colab начнёт конфликтовать с версиями, проще перезапустить runtime и выполнить ячейки заново.

In [ ]:
%%capture
!pip install -q -r requirements.txt

In [ ]:
# Быстрая проверка, что скрипты хотя бы импортируются/компилируются.
!python -m compileall -q src
print('compileall: ok')

## 3. Быстрый smoke-test без COCO

Этот блок нужен, чтобы показать: структура датасета, COCO-json, подсчёт классов и служебные скрипты работают даже без большого датасета.

Это не финальная метрика. Это только проверка пайплайна.

In [ ]:
!python src/make_toy_coco.py --out-root data/toy_coco
!python src/pick_rare_classes.py --data-root data/toy_coco --out-json reports/toy_class_distribution.json

import json
from pathlib import Path

toy_stats = json.loads(Path('reports/toy_class_distribution.json').read_text())
toy_stats

## 4. Подключение COCO 2017

Ожидаемая структура:

```text
data/coco/
  train2017/
  val2017/
  annotations/
    instances_train2017.json
    instances_val2017.json
```

В Colab удобнее положить COCO в Google Drive и смонтировать диск. Если датасет уже загружен в `/content/data/coco`, этот блок можно пропустить.

In [ ]:
from google.colab import drive
from pathlib import Path

MOUNT_DRIVE = False  # поменять на True, если COCO лежит на Google Drive

if MOUNT_DRIVE:
    drive.mount('/content/drive')
    # пример: COCO_ROOT = Path('/content/drive/MyDrive/datasets/coco')

COCO_ROOT = Path('/content/data/coco')
print('COCO_ROOT =', COCO_ROOT)
print('annotations exist:', (COCO_ROOT / 'annotations' / 'instances_train2017.json').exists())

## 5. Создание COCO-subset на 10 классов

Я беру 10 классов, чтобы обучение было подъёмным для Colab, но задача оставалась object detection, а не игрушкой. Размер subset можно уменьшить для быстрого прогона или увеличить для более честных метрик.

In [ ]:
# Можно менять под время и GPU.
CLASSES = 'person bicycle car motorcycle bus truck cat dog chair bottle'
MAX_TRAIN_IMAGES = 1200
MAX_VAL_IMAGES = 300

assert (COCO_ROOT / 'annotations' / 'instances_train2017.json').exists(), 'Сначала положи COCO в COCO_ROOT'

!python src/prepare_coco_subset.py \
  --coco-root {COCO_ROOT} \
  --out-root data/coco10 \
  --classes {CLASSES} \
  --max-train-images {MAX_TRAIN_IMAGES} \
  --max-val-images {MAX_VAL_IMAGES} \
  --link-mode copy

!python src/pick_rare_classes.py --data-root data/coco10 --out-json reports/class_distribution.json

In [ ]:
import json
import pandas as pd
from pathlib import Path

stats_path = Path('reports/class_distribution.json')
stats = json.loads(stats_path.read_text())

rows = []
for split, split_stats in stats.items():
    if isinstance(split_stats, dict):
        for cls, count in split_stats.items():
            rows.append({'split': split, 'class': cls, 'objects': count})

pd.DataFrame(rows).sort_values(['split', 'objects']).head(30)

## 6. Обучение DETR

Для защиты можно показать запуск на 1 эпоху, а финальные результаты заранее получить на 5–10 эпохах. Важно сохранить артефакты: `train_losses.csv`, `val_metrics.csv`, `checkpoints/best.pt`, `profiler/`.

In [ ]:
# Настройки запуска.
# Для демонстрации поставь EPOCHS=1. Для финального результата лучше 5-10.
EPOCHS = 5
BATCH_SIZE = 2
NUM_WORKERS = 2
RUN_DIR = 'runs/detr_coco10'

!python src/train_detr.py \
  --data-root data/coco10 \
  --model-name facebook/detr-resnet-50 \
  --output-dir {RUN_DIR} \
  --epochs {EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --lr 1e-5 \
  --lr-backbone 1e-6 \
  --weight-decay 1e-4 \
  --num-workers {NUM_WORKERS} \
  --profile

## 7. TensorBoard

Здесь удобно показать преподавателю динамику loss и profiler trace. Если trace не виден сразу, обычно помогает обновить TensorBoard или открыть вкладку `Profile`.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/detr_coco10/tb

## 8. Метрики и loss-графики

Смотрю не только итоговый `mAP`, но и динамику loss: у DETR отдельно важны classification loss, bbox L1 loss и GIoU loss.

In [ ]:
import pandas as pd
from pathlib import Path

loss_csv = Path(RUN_DIR) / 'train_losses.csv'
metrics_csv = Path(RUN_DIR) / 'val_metrics.csv'

if loss_csv.exists():
    display(pd.read_csv(loss_csv).tail())
else:
    print('Пока нет train_losses.csv')

if metrics_csv.exists():
    display(pd.read_csv(metrics_csv).tail())
else:
    print('Пока нет val_metrics.csv')

In [ ]:
!python src/plot_losses.py \
  --log-csv runs/detr_coco10/train_losses.csv \
  --out-dir reports/figures

from IPython.display import Image, display
from pathlib import Path

for img in sorted(Path('reports/figures').glob('*.png')):
    print(img)
    display(Image(filename=str(img)))

## 9. Визуализация предсказаний

Картинки с bbox нужны не для красоты, а чтобы увидеть типичные ошибки: мелкие объекты, перекрытия, похожие классы, ложные срабатывания.

In [ ]:
!python src/visualize_predictions.py \
  --data-root data/coco10 \
  --checkpoint runs/detr_coco10/checkpoints/best.pt \
  --out-dir reports/figures/predictions \
  --num-images 24 \
  --score-threshold 0.5

In [ ]:
from IPython.display import Image, display
from pathlib import Path

pred_imgs = sorted(Path('reports/figures/predictions').glob('*.jpg'))[:12]
if not pred_imgs:
    pred_imgs = sorted(Path('reports/figures/predictions').glob('*.png'))[:12]

for p in pred_imgs:
    print(p.name)
    display(Image(filename=str(p), width=650))

## 10. Error analysis

Разделяю ошибки на несколько типов:

- `classification_error`: объект найден, но класс перепутан;
- `localization_error`: класс верный, но bbox плохо попал;
- `missed_object`: объект был в разметке, но модель его пропустила;
- `false_positive`: модель нашла лишний объект.

Это сильный блок для защиты: он показывает не только число mAP, но и понимание поведения модели.

In [ ]:
!python src/error_analysis.py \
  --data-root data/coco10 \
  --checkpoint runs/detr_coco10/checkpoints/best.pt \
  --out-dir reports/error_analysis \
  --score-threshold 0.5 \
  --iou-threshold 0.5

In [ ]:
import json
import pandas as pd
from pathlib import Path

summary_path = Path('reports/error_analysis/summary.json')
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    display(pd.DataFrame([summary]))

for csv_name in ['errors.csv', 'confusion.csv']:
    p = Path('reports/error_analysis') / csv_name
    if p.exists():
        print('\n', csv_name)
        display(pd.read_csv(p).head(20))

## 11. HTML-отчёт для показа

Этот отчёт собирает основные артефакты в одну страницу: метрики, loss-графики, prediction gallery, error gallery. Его удобно открыть на защите вместо прыжков по папкам.

In [ ]:
!python src/make_demo_report.py \
  --run-dir runs/detr_coco10 \
  --reports-dir reports \
  --out-file reports/demo_report.html

from IPython.display import IFrame, display

display(IFrame('reports/demo_report.html', width='100%', height=900))

# Часть 2. Синтетика через Stable Diffusion + ControlNet

Идея: взять редкие классы, сгенерировать дополнительные изображения и проверить, помогает ли это классификатору на crop-ах.

Важно: синтетика не обязана улучшить качество. Если картинки слишком чистые или отличаются от COCO по домену, качество может даже упасть. Поэтому здесь главное — честный ablation.

## 12. Crop-ы объектов из COCO-subset

Извлекаю объекты по bbox и превращаю detection-разметку в классификационный датасет. Так проще и быстрее проверить эффект синтетики.

In [ ]:
!python src/extract_classification_crops.py \
  --data-root data/coco10 \
  --out-root data/crops10 \
  --min-area 1024 \
  --max-per-class 700

!find data/crops10 -maxdepth 2 -type d | sort | head -40

## 13. Генерация синтетики

Блок тяжёлый: для него точно нужен GPU. На защите можно показать уже сгенерированные примеры, а не ждать генерацию вживую.

По умолчанию генерация выключена, чтобы случайно не тратить время Colab.

In [ ]:
GENERATE_SYNTHETIC = False
SYNTH_CLASSES = 'bicycle motorcycle truck'  # лучше брать редкие классы из class_distribution.json
NUM_SYNTH_PER_CLASS = 40

if GENERATE_SYNTHETIC:
    !python src/generate_controlnet_synthetic.py \
      --input-root data/crops10/train \
      --output-root data/synth10 \
      --classes {SYNTH_CLASSES} \
      --num-per-class {NUM_SYNTH_PER_CLASS} \
      --steps 25
else:
    print('Генерация выключена. Для полного запуска поставь GENERATE_SYNTHETIC=True.')

In [ ]:
from IPython.display import Image, display
from pathlib import Path

synth_root = Path('data/synth10')
if synth_root.exists():
    imgs = []
    for ext in ('*.jpg', '*.png', '*.jpeg'):
        imgs.extend(synth_root.glob(f'*/*{ext[1:]}'))
    for p in sorted(imgs)[:12]:
        print(p)
        display(Image(filename=str(p), width=300))
else:
    print('Папки data/synth10 пока нет. Можно либо сгенерировать, либо загрузить заранее подготовленную синтетику.')

## 14. Ablation: real vs real + synthetic

Здесь сравниваю классификатор на crop-ах:

- baseline: только реальные crop-ы;
- experiment: реальные crop-ы + синтетика.

Итоговая таблица нужна в отчёт.

In [ ]:
RUN_ABLATION = False  # включить после генерации/загрузки синтетики

if RUN_ABLATION:
    !python src/train_classifier_ablation.py \
      --real-root data/crops10 \
      --synthetic-root data/synth10 \
      --output-dir runs/synthetic_ablation \
      --epochs 8
else:
    print('Ablation выключен. Включи RUN_ABLATION=True, когда будет data/synth10.')

In [ ]:
from pathlib import Path
import pandas as pd

ablation_csv = Path('runs/synthetic_ablation/ablation_metrics.csv')
if ablation_csv.exists():
    display(pd.read_csv(ablation_csv))
else:
    print('Пока нет ablation_metrics.csv')

# Короткие тезисы для защиты

Можно говорить так:

1. Я взял COCO-subset на 10 классов, потому что полный COCO слишком тяжёлый для учебного fine-tuning в Colab.
2. В качестве детектора использовал DETR: он предсказывает фиксированный набор object queries и обучается через bipartite matching, поэтому не требует anchor boxes и NMS в классическом виде.
3. Логировал отдельно classification loss, bbox L1 loss и GIoU loss, потому что итоговый loss сам по себе плохо объясняет, где именно модель ошибается.
4. Кроме mAP/mAP50 сделал qualitative analysis: картинки с bbox и отдельный разбор ошибок.
5. В error analysis разделил ошибки на classification, localization, missed objects и false positives.
6. Для синтетики взял редкие классы, потому что именно там дополнительная вариативность потенциально полезнее.
7. Я не утверждаю, что синтетика всегда улучшает качество. Поэтому сделал ablation `real` vs `real + synthetic` и смотрю на таблицу, а не только на красивые картинки.

Главный вывод лучше формулировать после реального запуска: модель научилась находить крупные частые объекты лучше, а основные проблемы остались на мелких объектах, перекрытиях и похожих классах.

# Что сдавать в GitHub

Минимальный набор артефактов:

```text
src/
README.md
requirements.txt
configs/
notebooks/cv_hw_colab_demo.ipynb
reports/REPORT_TEMPLATE.md
reports/demo_report.html
reports/figures/
reports/error_analysis/
runs/detr_coco10/train_losses.csv
runs/detr_coco10/val_metrics.csv
runs/detr_coco10/checkpoints/best.pt   # если размер позволяет
runs/synthetic_ablation/ablation_metrics.csv
```

Большие данные COCO лучше не класть в репозиторий. В README достаточно описать, куда их положить.